# H3 routing evaluation, focused on the Sveta Troitsa neighborhood (notebook 14)

Notebook 13's first systematic H3 pass sampled 40 fully random
origin-destination pairs across the *entire* Sofia graph (27,429 edges) and
found that 0 crossed any of the 4 road segments that had real detection
coverage at the time -- not an algorithm failure, a data-coverage problem.

This notebook fixes that by concentrating both the data collection and the
route sampling in one real neighborhood: **Sveta Troitsa**. 14 new,
genuinely geotagged photos (phone GPS, not the CVAT-exported training set,
which had its EXIF stripped) were taken across this neighborhood's main
streets and processed through the exact same pipeline the Streamlit app
uses (`scripts/batch_upload_sveta_troitsa.py`), joining 3 already-saved
photos from the same area -- **17 real observations, 16 distinct road
segments with recorded risk**, up from 4 segments city-wide before.

Route sampling here is restricted to graph nodes actually inside the
neighborhood's bounding box (derived from the real GPS extent of the 17
photos, not eyeballed from a map), so the generated test routes have an
actual chance of crossing the segments that now have real risk data.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import folium
import networkx as nx
from scipy import stats

sys.path.append("..")
from src.app import database as db
from src.geospatial.network import get_sofia_graph
from src.routing.router import compare_routes

db.init_db()

## Step 1 -- define the neighborhood and pull live risk data from the database

Two things worth being explicit about:
- The bounding box comes from the **actual GPS coordinates** of the 17
  Sveta Troitsa photos (with a ~110m margin), not an eyeballed map region.
- Segment risk is read **directly from the live app database**
  (`road_segments` table), not from `outputs/geo/segment_risk.csv` --
  those are two separate storage tracks in this project (see
  `thesis/model_training_results_and_discussion.md` section 8), and the
  CSV was never regenerated after these new uploads. Reading the database
  directly is the correct, current source of truth here.

In [2]:
graph = get_sofia_graph()

MARGIN_DEG = 0.001  # ~110m
LAT_MIN, LAT_MAX = 42.702118 - MARGIN_DEG, 42.713770 + MARGIN_DEG
LON_MIN, LON_MAX = 23.285544 - MARGIN_DEG, 23.299341 + MARGIN_DEG

neighborhood_nodes = [
    n for n, d in graph.nodes(data=True)
    if LAT_MIN <= d["y"] <= LAT_MAX and LON_MIN <= d["x"] <= LON_MAX
]
print(f"Sveta Troitsa bounding box: lat [{LAT_MIN:.5f}, {LAT_MAX:.5f}], lon [{LON_MIN:.5f}, {LON_MAX:.5f}]")
print(f"Nodes inside the neighborhood: {len(neighborhood_nodes)} (of {graph.number_of_nodes()} total)")

# Build segment_risk directly from the live DB, in the format compare_routes() expects.
db_segments = db.get_all_road_segments()
segment_risk = {
    f"{s['edge_u']}_{s['edge_v']}_{s['edge_key']}": {
        "risk_density": s["risk_density"],
        "total_risk": s["aggregated_risk"],
    }
    for s in db_segments
}
print(f"Segments with recorded risk (live DB): {len(segment_risk)}")

Sveta Troitsa bounding box: lat [42.70112, 42.71477], lon [23.28454, 23.30034]
Nodes inside the neighborhood: 206 (of 11363 total)
Segments with recorded risk (live DB): 16


## Step 2 -- generate independent test routes within the neighborhood

In [3]:
import random
random.seed(42)

N_ROUTES = 40
route_records = []
attempts = 0

while len(route_records) < N_ROUTES and attempts < N_ROUTES * 20:
    attempts += 1
    u, v = random.sample(neighborhood_nodes, 2)
    try:
        path_len = nx.shortest_path_length(graph, u, v, weight="length")
    except nx.NetworkXNoPath:
        continue
    if path_len < 100:  # skip trivially short pairs
        continue

    origin = (graph.nodes[u]["y"], graph.nodes[u]["x"])
    dest = (graph.nodes[v]["y"], graph.nodes[v]["x"])
    comparison = compare_routes(graph, origin, dest, segment_risk=segment_risk)
    route_records.append(comparison)

print(f"Generated {len(route_records)} independent test routes ({attempts} attempts).")

Generated 40 independent test routes (41 attempts).


## Step 3 -- aggregate metrics (matching the app spec's H3 required outputs)

In [4]:
distance_increase_pct = []
exposure_reduction_pct = []
n_with_any_hazard = 0
n_satisfying_10pct = 0
shortest_exposures, safe_exposures = [], []

for c in route_records:
    s, r = c.shortest_route, c.risk_aware_route
    d_pct = ((r.distance_m - s.distance_m) / s.distance_m * 100) if s.distance_m > 0 else 0.0
    distance_increase_pct.append(d_pct)
    n_satisfying_10pct += int(d_pct <= 10.0)

    if s.total_risk_encountered > 0:
        n_with_any_hazard += 1
        e_pct = (s.total_risk_encountered - r.total_risk_encountered) / s.total_risk_encountered * 100
        exposure_reduction_pct.append(e_pct)

    shortest_exposures.append(s.total_risk_encountered)
    safe_exposures.append(r.total_risk_encountered)

shortest_exposures = np.array(shortest_exposures)
safe_exposures = np.array(safe_exposures)

print(f"Routes generated:                            {len(route_records)}")
print(f"Routes crossing >=1 known hazardous segment:    {n_with_any_hazard} ({n_with_any_hazard/len(route_records)*100:.0f}%)")
print(f"Median distance increase (all routes):          {np.median(distance_increase_pct):.2f}%")
print(f"Mean distance increase (all routes):             {np.mean(distance_increase_pct):.2f}%")
print(f"% satisfying the <=10% distance constraint:       {n_satisfying_10pct/len(route_records)*100:.0f}%")
if exposure_reduction_pct:
    print(f"Mean exposure reduction (hazard-crossing routes): {np.mean(exposure_reduction_pct):.1f}%")
    print(f"Median exposure reduction (hazard-crossing routes): {np.median(exposure_reduction_pct):.1f}%")
else:
    print("No generated route crossed a known hazardous segment.")

Routes generated:                            40
Routes crossing >=1 known hazardous segment:    18 (45%)
Median distance increase (all routes):          0.00%
Mean distance increase (all routes):             1.50%
% satisfying the <=10% distance constraint:       95%
Mean exposure reduction (hazard-crossing routes): 45.6%
Median exposure reduction (hazard-crossing routes): 37.2%


## Step 4 -- paired significance test: shortest-route exposure vs. risk-aware-route exposure

Same explicit zero-variation check as notebook 13 (rather than relying on
scipy to raise an error, which it doesn't reliably do).

In [5]:
if np.all(shortest_exposures == safe_exposures):
    print(f"Every route has identical shortest/risk-aware exposure (0 of {len(route_records)} crossed "
          "a hazardous segment) -- not enough variation for a paired significance test, even within "
          "the neighborhood.")
    h3_stat_verdict = "NOT TESTABLE -- insufficient risk-data coverage even within the neighborhood"
else:
    wilcoxon_h3 = stats.wilcoxon(shortest_exposures, safe_exposures, alternative="greater")
    print(f"Wilcoxon signed-rank test (H0: no difference, H1: shortest-route exposure > risk-aware): "
          f"statistic={wilcoxon_h3.statistic:.1f}, p={wilcoxon_h3.pvalue:.4f}")
    h3_stat_verdict = "SUPPORTED (p < 0.05)" if wilcoxon_h3.pvalue < 0.05 else "NOT SUPPORTED (insufficient evidence)"

print(f"\nH3 (statistical mechanism check, Sveta Troitsa): {h3_stat_verdict}")

Wilcoxon signed-rank test (H0: no difference, H1: shortest-route exposure > risk-aware): statistic=66.0, p=0.0017

H3 (statistical mechanism check, Sveta Troitsa): SUPPORTED (p < 0.05)


## Step 5 -- visualize the neighborhood's segment risk and a few example routes

In [6]:
center = ((LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2)
fmap = folium.Map(location=center, zoom_start=15, tiles="OpenStreetMap")

CATEGORY_COLOR = {"low": "green", "medium": "orange", "high": "red"}

for seg in db_segments:
    try:
        edge_data = graph.get_edge_data(seg["edge_u"], seg["edge_v"], seg["edge_key"])
        geometry = edge_data.get("geometry")
        if geometry is not None:
            coords = [(lat, lon) for lon, lat in geometry.coords]
        else:
            coords = [
                (graph.nodes[seg["edge_u"]]["y"], graph.nodes[seg["edge_u"]]["x"]),
                (graph.nodes[seg["edge_v"]]["y"], graph.nodes[seg["edge_v"]]["x"]),
            ]
    except Exception:
        continue
    folium.PolyLine(
        coords, color=CATEGORY_COLOR.get(seg["risk_category"], "blue"), weight=6, opacity=0.8,
        tooltip=f"{seg['road_name'] or 'unnamed'} -- {seg['risk_category']} (risk {seg['aggregated_risk']:.2f}, "
                f"{seg['observation_count']} observation(s))",
    ).add_to(fmap)

# A handful of example generated routes, shortest (blue) vs risk-aware (green dashed),
# limited to ones that actually crossed a hazard for a meaningful visual.
example_routes = [c for c in route_records if c.shortest_route.total_risk_encountered > 0][:5]
for c in example_routes:
    folium.PolyLine(c.shortest_route.coordinates, color="blue", weight=3, opacity=0.6, tooltip="shortest").add_to(fmap)
    folium.PolyLine(c.risk_aware_route.coordinates, color="green", weight=3, opacity=0.6, dash_array="6",
                     tooltip="risk-aware").add_to(fmap)

fmap.save("sveta_troitsa_routing_map.html")
print(f"Map saved to notebooks/sveta_troitsa_routing_map.html ({len(example_routes)} example hazard-crossing routes shown)")
fmap

Map saved to notebooks/sveta_troitsa_routing_map.html (5 example hazard-crossing routes shown)


## Notes

- 16 real road segments now carry detection-based risk in the live
  database (up from 4), all within or adjacent to Sveta Troitsa -- a
  genuine, if geographically scoped, improvement in data coverage.
- This evaluation is explicitly scoped to one neighborhood, chosen for the
  feasibility of collecting real, geotagged ground-truth photos within the
  project's timeframe -- not claimed as representative of the whole city
  (same kind of explicit scoping decision as the project's existing
  Sofia-only / 4-class-only choices).
- If this batch still doesn't produce enough hazard-crossing routes for a
  significant paired test, the honest next lever is more photos in the
  same neighborhood (denser coverage of its side streets), not a larger
  or more scattered set across the city.